[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C38_Frameworks_Accel_Course/05_profiling_debugging/05_profiling_debugging.ipynb)

# 05 · 性能剖析与调试（玩具 profiler + NaN 溯源）

目标：造一台简易「听诊器」——用 numpy 写一个**玩具 profiler**（计时 + 按算子归因 + 显存追踪），用它**定位热点与显存峰值**；再写一个 **NaN 溯源器**定位首个出错算子；最后看**浮点不结合**如何破坏可复现。

路线：计时器（warmup+取最小）→ 按算子归因 → 显存追踪+峰值 → NaN 溯源 → 确定性(浮点不结合) → 优化闭环 → ✏️ 练习（计时统计 / 瓶颈识别 / NaN 定位 / 显存峰值）→ 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：优化闭环 = **测量 → 定位瓶颈 → 优化 → 再测量验证**。先测量，别猜。

## 1 · 计时器：warmup + 多次取最小

正确计时要：**预热**（丢弃首次的一次性开销）、**多次取最小**（最小值最接近无干扰真实耗时）。
（GPU 上还要 `synchronize`——见讲解；CPU 无此问题。）写一个计时函数体现这些原则。

In [ ]:
import numpy as np, time
rng = np.random.default_rng(0)

def benchmark(fn, warmup=2, repeats=5):
    '''预热 warmup 次(丢弃)，再测 repeats 次取最小耗时(秒)。'''
    for _ in range(warmup):
        fn()                              # 预热：触发一次性开销
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)                     # 取最小(最接近真实，避开偶发卡顿)

# 测两个不同规模的 matmul，看耗时随规模增长
def make_matmul(n):
    A = rng.standard_normal((n, n)); B = rng.standard_normal((n, n))
    return lambda: A @ B

t_small = benchmark(make_matmul(64))
t_big   = benchmark(make_matmul(256))
print(f'matmul 64x64  : {t_small*1e6:8.1f} us')
print(f'matmul 256x256: {t_big*1e6:8.1f} us')
# 256 是 64 的 4 倍边长 -> ~4^3=64 倍计算量, 应明显更慢
assert t_big > t_small, '更大的 matmul 应更慢'
assert t_small > 0
print('✅ 计时器：warmup + 取最小，量出耗时随规模增长')

## 2 · 玩具 profiler：按算子归因

核心产出是**按算子聚合的耗时表**。用一个上下文管理器 `prof.timer(name)` 给代码段打标签、累加耗时与调用次数，最后产出报告（== `torch.profiler.record_function` 的精神）。

In [ ]:
from contextlib import contextmanager

class Profiler:
    def __init__(self):
        self.times = {}      # name -> 累计耗时
        self.counts = {}     # name -> 调用次数
    @contextmanager
    def timer(self, name):
        t0 = time.perf_counter()
        try:
            yield
        finally:
            dt = time.perf_counter() - t0
            self.times[name] = self.times.get(name, 0.0) + dt
            self.counts[name] = self.counts.get(name, 0) + 1
    def report(self):
        total = sum(self.times.values()) or 1e-12
        rows = sorted(self.times.items(), key=lambda kv: -kv[1])
        print(f"{'算子':<14}{'耗时(ms)':>10}{'占比':>8}{'次数':>7}")
        for name, t in rows:
            print(f'{name:<14}{t*1e3:>10.3f}{t/total:>7.1%}{self.counts[name]:>7}')
        return rows, total
    def hotspot(self):
        return max(self.times.items(), key=lambda kv: kv[1])[0]

# 模拟一个有 matmul / softmax / layernorm 的前向，profile 它
# 用 512x512：matmul 是 O(n^3)、softmax/layernorm 是 O(n^2)，在这个规模下 matmul 才真正成为热点
# （128 这种小矩阵 + 快 BLAS 下，逐元素的 softmax 反而更慢，profiler 会误指）。
prof = Profiler()
n = 512
X = rng.standard_normal((n, n))
W = rng.standard_normal((n, n))
for _ in range(8):
    with prof.timer('matmul'):
        Y = X @ W @ W            # 故意做两次 matmul（重活，O(n^3)）
    with prof.timer('softmax'):
        e = np.exp(Y - Y.max(axis=1, keepdims=True)); P = e / e.sum(axis=1, keepdims=True)
    with prof.timer('layernorm'):
        Z = (Y - Y.mean(1, keepdims=True)) / (Y.std(1, keepdims=True) + 1e-5)
rows, total = prof.report()
hot = prof.hotspot()
print(f'\n热点算子 = {hot}')
assert hot == 'matmul', 'matmul(两次,大)应是热点'
assert prof.counts['matmul'] == 8
print('✅ profiler 按算子归因，定位热点 = matmul（优化它收益最大）')

## 3 · 显存追踪器：峰值与 OOM

维护一个运行中的显存计数器：分配张量就加、释放就减，记录历史峰值。
模拟一次训练 step 的显存变化（前向激活累积→反向开始达峰→逐步释放），定位峰值、判断是否 OOM。

In [ ]:
class MemTracker:
    def __init__(self):
        self.current = 0
        self.peak = 0
        self.timeline = []
    def alloc(self, nbytes, tag=''):
        self.current += nbytes
        self.peak = max(self.peak, self.current)
        self.timeline.append((tag, self.current))
    def free(self, nbytes, tag=''):
        self.current -= nbytes
        self.timeline.append((tag, self.current))
    def will_oom(self, capacity):
        return self.peak > capacity

# 模拟: 常驻(参数+优化器) + 前向逐层激活累积 + 反向逐层释放
def simulate_step(n_layers, act_size, resident, use_checkpoint=False):
    m = MemTracker()
    m.alloc(resident, 'resident(参数+优化器)')
    if use_checkpoint:
        # 只存 sqrt(L) 个检查点的激活
        import math
        kept = max(1, int(math.sqrt(n_layers)))
        for i in range(kept):
            m.alloc(act_size, f'ckpt_act{i}')
    else:
        for i in range(n_layers):                 # 前向：存全部激活
            m.alloc(act_size, f'act{i}')
    peak_at_backward = m.peak
    # 反向：逐步释放
    while m.current > resident:
        m.free(act_size, 'free_act')
    return m, peak_at_backward

resident = 400_000_000      # 400 MB 常驻
act = 50_000_000            # 每层激活 50 MB
m_full, peak_full = simulate_step(16, act, resident, use_checkpoint=False)
m_ckpt, peak_ckpt = simulate_step(16, act, resident, use_checkpoint=True)
print(f'全存激活   峰值 = {peak_full/1e6:7.0f} MB')
print(f'checkpoint 峰值 = {peak_ckpt/1e6:7.0f} MB')
assert peak_full > peak_ckpt, 'checkpointing 应降低峰值'
# OOM 判断：假设卡只有 1 GB
cap = 1_000_000_000
assert m_full.will_oom(cap) and not m_ckpt.will_oom(cap)
print(f'\n容量 1GB: 全存 {"OOM!" if m_full.will_oom(cap) else "ok"}, checkpoint {"OOM!" if m_ckpt.will_oom(cap) else "ok"}')
print('✅ 显存追踪：定位峰值(反向开始)，checkpointing 把峰值压到容量内、避免 OOM')

## 4 · NaN 溯源器：定位第一个出错算子

NaN 会传染——找到**第一个**产生 NaN 的算子才是关键。写一个溯源器：在每个算子后检查输出，
第一个出现 NaN/Inf 的就是源头（== `torch.autograd.set_detect_anomaly` 的原理）。

In [ ]:
def check_tensor(name, x):
    '''检查张量是否被污染(NaN/Inf)。'''
    x = np.asarray(x)
    return bool(np.isnan(x).any() or np.isinf(x).any())

def run_with_nan_tracing(steps):
    '''steps: [(name, fn)]，fn 接受 env、返回输出。第一个产 NaN 的算子即源头。'''
    env = {}
    for name, fn in steps:
        out = fn(env)
        env[name] = out
        if check_tensor(name, out):
            return name, env          # 定位首个 NaN 源
    return None, env

# 构造一个【故意含 NaN 源】的流水：probs 有个 0，log(0) = -inf -> 后续 NaN
probs = np.array([0.5, 0.5, 0.0, 0.3])      # 第三个是 0
steps = [
    ('matmul',   lambda e: rng.standard_normal((4, 4)) @ np.ones((4, 1))),
    ('relu',     lambda e: np.maximum(e['matmul'], 0)),
    ('log_probs',lambda e: np.log(probs)),       # log(0) = -inf  <- 元凶在这
    ('weighted', lambda e: e['log_probs'] * 2.0),# -inf 传染
    ('loss',     lambda e: e['weighted'].sum()),
]
culprit, env = run_with_nan_tracing(steps)
print(f'首个产生 NaN/Inf 的算子 = {culprit!r}')
print(f'它的输出 = {env[culprit]}')
assert culprit == 'log_probs', '应定位到 log(0) 那一步'
# 在它之前的算子都是干净的
assert not check_tensor('matmul', env['matmul'])
print('✅ NaN 溯源：精确定位到 log(0)，而非下游一片 NaN —— 找到零号病人')

**对策**：定位到 `log(0)` 后，对策清晰——用 `log_softmax`、或 `log(x + eps)`。验证加 eps 后流水干净。

In [ ]:
# 修复：log(probs + eps)
eps = 1e-9
steps_fixed = steps[:2] + [
    ('log_probs', lambda e: np.log(probs + eps)),    # 加 eps
    ('weighted',  lambda e: e['log_probs'] * 2.0),
    ('loss',      lambda e: e['weighted'].sum()),
]
culprit2, env2 = run_with_nan_tracing(steps_fixed)
assert culprit2 is None, '加 eps 后不应再有 NaN'
assert not check_tensor('loss', env2['loss'])
print(f'修复后 loss = {env2["loss"]:.4f}, 无 NaN ✅')
print('✅ 定位 -> 对症(加 eps) -> 验证干净。这就是 NaN 调试的闭环。')

## 5 · 确定性：浮点不结合如何破坏可复现

并行规约的累加**顺序不定**，而浮点加法**不满足结合律** `(a+b)+c ≠ a+(b+c)`，于是结果有微小差异、不可复现。
用「不同顺序求和」模拟，展示差异；再用固定顺序恢复确定性。

In [ ]:
# 一组数值跨度很大的数（最能暴露浮点不结合）
vals = np.array([1e8, 1.0, -1e8, 1e-3, 2e-3], dtype=np.float64)

# 不同求和顺序 -> 不同结果（模拟并行规约的不确定顺序）
order1 = vals.copy()
order2 = vals[::-1].copy()
order3 = np.array([vals[0], vals[2], vals[1], vals[3], vals[4]])  # 另一种顺序
s1 = 0.0
for v in order1: s1 += v
s2 = 0.0
for v in order2: s2 += v
s3 = 0.0
for v in order3: s3 += v
print(f'顺序1 求和 = {s1!r}')
print(f'顺序2 求和 = {s2!r}')
print(f'顺序3 求和 = {s3!r}  (先抵消大数)')
# 至少有两个顺序结果不同 -> 不可复现
assert not (s1 == s2 == s3), '不同顺序应给出不同结果（浮点不结合）'
print('\n❌ 不同顺序结果不同 -> 并行规约不可复现（浮点加法不结合）')

# 恢复确定性：强制固定顺序（如总是升序加，或用 numpy 确定性的 sum）
det_a = float(np.sort(vals).sum())
det_b = float(np.sort(vals).sum())
assert det_a == det_b, '固定顺序 -> 逐位可复现'
print(f'固定顺序(排序后求和)两次: {det_a!r} == {det_b!r}')
print('✅ 确定性 = 固定随机种子 + 固定规约顺序（代价:常更慢）')

## 6 · 优化闭环：测量→定位→优化→再测量

把前面的工具串成一个**完整闭环**：profile 找热点 → 优化它 → 再 profile 确认变快 → 对拍确认仍正确。
演示一个真实场景：发现重复计算、消除它、验证。

In [ ]:
# 场景：一段代码里 X@W 被算了两次（浪费）
X = rng.standard_normal((200, 200)); W = rng.standard_normal((200, 200))

def slow_version():
    a = (X @ W).sum()        # 第一次 X@W
    b = (X @ W).mean()       # 又算一次 X@W（浪费！）
    return a + b

def fast_version():
    XW = X @ W               # 只算一次，复用（CSE，模块 02）
    return XW.sum() + XW.mean()

# 1) 测量
t_slow = benchmark(slow_version, repeats=10)
t_fast = benchmark(fast_version, repeats=10)
print(f'优化前: {t_slow*1e3:.3f} ms')
print(f'优化后: {t_fast*1e3:.3f} ms  (加速 {t_slow/t_fast:.2f}x)')
# 2) 对拍：优化不能改变结果！
assert np.isclose(slow_version(), fast_version()), '优化必须保结果'
# 3) 确认变快
assert t_fast < t_slow, '消除重复计算应更快'
print('✅ 优化闭环：测量发现浪费 -> 消除重复(复用 X@W) -> 更快且结果不变')

---
## ✏️ 练习 1：给 profiler 加「平均耗时」统计

给 `Profiler` 加一个方法 `avg_time(name)`：返回某算子的**平均**单次耗时（总耗时 / 调用次数，毫秒）。

实现 `profiler_avg_ms(prof, name)`（独立函数，读 `prof.times`/`prof.counts`，返回毫秒）。

In [ ]:
def profiler_avg_ms(prof, name):
    # TODO: 返回 prof.times[name] / prof.counts[name] * 1000（毫秒）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
# 复用上面 worked 2 的 prof（matmul 调用了 8 次）
avg = profiler_avg_ms(prof, 'matmul')
expected = prof.times['matmul'] / prof.counts['matmul'] * 1000
assert np.isclose(avg, expected), '平均耗时 = 总耗时/次数'
assert avg > 0
print(f'✅ 练习 1 通过：matmul 平均单次耗时 = {avg:.4f} ms')

## ✏️ 练习 2：根据 profile 报告判断瓶颈性质

给定一份 profile 报告（算子 → (占比, 调用次数)），判断**最大热点**的瓶颈性质：
- 占比最大且**调用次数多**（>100）→ 可能 `'overhead-bound'`（一堆小内核，该融合）；
- 占比最大且**调用次数少**（≤100）→ 可能 `'compute-or-memory-bound'`（单次贵的大算子）。

实现 `classify_bottleneck(report)`：`report` 是 `{name: (pct, count)}`，返回 `(热点名, 性质字符串)`。

In [ ]:
def classify_bottleneck(report):
    # TODO: 1) 找占比(pct)最大的算子；2) 按其 count 判断性质
    #       count > 100 -> 'overhead-bound'，否则 'compute-or-memory-bound'
    #       返回 (热点名, 性质)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 报告 A: 大量小 elementwise（overhead-bound）
report_a = {'matmul': (0.20, 4), 'add': (0.55, 500), 'mul': (0.25, 480)}
name_a, kind_a = classify_bottleneck(report_a)
assert name_a == 'add' and kind_a == 'overhead-bound', (name_a, kind_a)
# 报告 B: 一个大 matmul 主导（compute/memory-bound）
report_b = {'matmul': (0.75, 8), 'softmax': (0.15, 6), 'norm': (0.10, 12)}
name_b, kind_b = classify_bottleneck(report_b)
assert name_b == 'matmul' and kind_b == 'compute-or-memory-bound', (name_b, kind_b)
print('✅ 练习 2 通过：能据占比+次数定性瓶颈（overhead vs compute/memory）')

## ✏️ 练习 3：定位首个 NaN 并指出可能原因

扩展 NaN 溯源：实现 `trace_and_diagnose(steps)`，返回 `(首个NaN算子名, 它的输入是否也已含NaN)`。
若输入干净但输出 NaN → 是这个算子**自身**产生的（如 log(0)）；若输入已 NaN → 它只是被传染。

（`steps` 里每个 fn 接受 env；用 env 里它依赖的前序结果判断输入是否干净。简化：检查 env 中所有已算出的值。）

In [ ]:
def trace_and_diagnose(steps):
    # TODO: 跑流水；找到首个输出含 NaN/Inf 的算子 name；
    #       判断在它之前 env 里是否【已有】NaN(即它被传染) -> inputs_dirty=True
    #       返回 (name, inputs_dirty)。若无 NaN 返回 (None, False)。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
bad = np.array([1.0, 0.0, 2.0])
steps2 = [
    ('a', lambda e: np.array([1.0, 2.0, 3.0])),     # 干净
    ('b', lambda e: np.log(bad)),                   # log(0)=-inf, 自身产生
    ('c', lambda e: e['b'] + e['a']),               # 被 b 传染
]
name, dirty = trace_and_diagnose(steps2)
assert name == 'b', '首个 NaN 源是 b (log(0))'
assert dirty == False, 'b 之前 env 干净 -> 是 b 自身产生的，不是被传染'
print(f'✅ 练习 3 通过：首个 NaN 算子={name!r}，自身产生(输入干净)={not dirty}')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def profiler_avg_ms(prof, name):
    return prof.times[name] / prof.counts[name] * 1000.0

In [ ]:
# 练习 2 参考答案
def classify_bottleneck(report):
    name = max(report, key=lambda k: report[k][0])    # 占比最大
    pct, count = report[name]
    kind = 'overhead-bound' if count > 100 else 'compute-or-memory-bound'
    return name, kind

In [ ]:
# 练习 3 参考答案
def trace_and_diagnose(steps):
    env = {}
    for name, fn in steps:
        # 先看当前 env 是否已有 NaN（在算本步之前）
        env_dirty_before = any(check_tensor(k, v) for k, v in env.items())
        out = fn(env)
        env[name] = out
        if check_tensor(name, out):
            return name, env_dirty_before
    return None, False

---
## 🧪 真实数据胶囊：对照 torch.profiler

我们的玩具 profiler 和真实 `torch.profiler` 做的是同一件事：按算子归因耗时、找热点。
下面用我们的 profiler 分析一个小「Transformer block」前向，找出热点。

**装了 torch 才用 torch.profiler 对照；没装则用我们的 profiler（不阻断）。**

In [ ]:
# 用我们的 profiler 分析一个玩具 attention 前向
d, seq = 64, 32
Q = rng.standard_normal((seq, d)); Kk = rng.standard_normal((seq, d)); V = rng.standard_normal((seq, d))
prof2 = Profiler()
for _ in range(20):
    with prof2.timer('QK^T'):
        S = Q @ Kk.T / np.sqrt(d)
    with prof2.timer('softmax'):
        e = np.exp(S - S.max(1, keepdims=True)); A = e / e.sum(1, keepdims=True)
    with prof2.timer('AV'):
        O = A @ V
rows2, _ = prof2.report()
print(f'\n热点 = {prof2.hotspot()}')

**🧪 胶囊练习**：实现 `profile_attention()`：
- 若有 torch：用 `torch.profiler.profile` 跑同样的 attention，返回它是否成功产出了算子级 profile（True/False）；
- 若没 torch：返回我们的 `prof2` 是否成功识别了热点（`hotspot()` 非空且在三个算子里）。

学生骨架（不计入自动验证）：

In [ ]:
def profile_attention():
    # TODO: 有 torch -> torch.profiler.profile 跑 attention，返回 True；
    #       没 torch -> 返回 prof2.hotspot() in {'QK^T','softmax','AV'}
    raise NotImplementedError

In [ ]:
# 自测（学生填好后运行）
assert profile_attention() == True
print('✅ 胶囊通过：profiler(我们的 或 torch.profiler) 成功按算子归因、定位热点')

In [ ]:
# 📖 胶囊参考答案
def profile_attention():
    try:
        import torch
        from torch.profiler import profile, ProfilerActivity
        q = torch.randn(seq, d); k = torch.randn(seq, d); v = torch.randn(seq, d)
        with profile(activities=[ProfilerActivity.CPU]) as prof_t:
            s = q @ k.T / (d ** 0.5)
            a = torch.softmax(s, dim=1)
            o = a @ v
        return len(prof_t.key_averages()) > 0     # 成功产出算子级统计
    except Exception:
        return prof2.hotspot() in {'QK^T', 'softmax', 'AV'}

---
## 🔧 旁注：真实 profiler / anomaly 怎么用

我们手写的 profiler + NaN 溯源 + 显存追踪，在 PyTorch 里都有内建对应（对照，**不依赖即可读**）：

```python
import torch
from torch.profiler import profile, ProfilerActivity

# 1) 算子级 profiling（== 我们的 Profiler.report）
with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
    model(x)
print(prof.key_averages().table(sort_by='cuda_time_total'))   # 按算子耗时排序

# 2) 显存峰值（== 我们的 MemTracker.peak）
torch.cuda.reset_peak_memory_stats()
model(x); loss.backward()
print(torch.cuda.max_memory_allocated() / 1e9, 'GB peak')

# 3) NaN 溯源（== 我们的 run_with_nan_tracing）
with torch.autograd.set_detect_anomaly(True):   # 一出 NaN 就报错并指出算子
    loss.backward()

# 4) 确定性（== 我们的固定顺序）
torch.manual_seed(0); torch.use_deterministic_algorithms(True)
```

对应关系：`profiler.key_averages` ↔ 我们的归因表；`max_memory_allocated` ↔ MemTracker.peak；`set_detect_anomaly` ↔ NaN 溯源；`use_deterministic_algorithms` ↔ 固定规约顺序。

### 小结（也是全课收尾）
- **先测量，别猜**：性能直觉常错；用 profiler 把耗时/显存归因到算子。GPU 计时务必 `synchronize` + warmup + 取最小。
- **按算子归因**找热点（占比+次数），用**瓶颈三分法**(模块00)定性，性质决定用哪种武器(融合/精度/编译)。
- **显存追踪**定位峰值(反向开始)与 OOM；**NaN 溯源**找第一个出错算子(零号病人)。
- **确定性** = 固定种子 + 固定规约顺序（浮点不结合 → 顺序影响结果），代价是性能。
- **优化闭环**：测量→定位→优化→**再测量+对拍**。贯穿全课：先有 ground truth，再优化，优化后对拍。

**全课完结** 🎉 你已从零复刻了 autograd、编译器、函数变换、混合精度、profiler——框架的五大内核。回到 PyTorch/JAX，你看到的不再是黑箱。